#Assignment 3
#Anish Tammineedi
#11818745


In [7]:
# Get the datasets
!!/usr/bin/curl --output test.dat https://raw.githubusercontent.com/huangyanann/CSCE5218/main/test_small.txt
!!/usr/bin/curl --output train.dat https://raw.githubusercontent.com/huangyanann/CSCE5218/main/train.txt


['  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current',
 '                                 Dload  Upload   Total   Spent    Left  Speed',
 '',
 '  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0',
 '100 11645  100 11645    0     0   167k      0 --:--:-- --:--:-- --:--:--  169k']

In [8]:
# Take a peek at the datasets
!head train.dat
!head test.dat

A1	A2	A3	A4	A5	A6	A7	A8	A9	A10	A11	A12	A13	
1	1	0	0	0	0	0	0	1	1	0	0	1	0
0	0	1	1	0	1	1	0	0	0	0	0	1	0
0	1	0	1	1	0	1	0	1	1	1	0	1	1
0	0	1	0	0	1	0	1	0	1	1	1	1	0
0	1	0	0	0	0	0	1	1	1	1	1	1	0
0	1	1	1	0	0	0	1	0	1	1	0	1	1
0	1	1	0	0	0	1	0	0	0	0	0	1	0
0	0	0	1	1	0	1	1	1	0	0	0	1	0
0	0	0	0	0	0	1	0	1	0	1	0	1	0
X1	X2	X3
1	1	1	1
0	0	1	1
0	1	1	0
0	1	1	0
0	1	1	0
0	1	1	0
0	1	1	0
0	1	1	0
1	1	1	1


#Perceptron Model

In [9]:
import math
import itertools
import re


# Corpus reader, all columns but the last one are coordinates;
# the last column is the label
def read_data(file_name):
    f = open(file_name, 'r')

    data = []
    # Discard header line
    f.readline()
    for instance in f.readlines():
        if not re.search('\t', instance):
            continue
        instance = list(map(int, instance.strip().split('\t')))
        # Add a dummy input so that w0 becomes the bias
        instance = [-1] + instance
        data += [instance]
    return data


def dot_product(array1, array2):
    # Return dot product of array1 and array2
    return sum(a*b for a, b in zip(array1, array2[:-1]))


def sigmoid(x):
    # Return output of sigmoid function
    return 1 / (1 + math.exp(-x))


# The output of the model
def output(weight, instance):
    # Return sigmoid(dot_product)
    return sigmoid(dot_product(weight, instance))


# Predict the label of an instance
def predict(weights, instance):
    # Output 1 if sigmoid result >= 0.5 else 0
    return 1 if output(weights, instance) >= 0.5 else 0


# Accuracy = percent of correct predictions
def get_accuracy(weights, instances):
    correct = sum([1 if predict(weights, instance) == instance[-1] else 0
                   for instance in instances])
    return correct * 100 / len(instances)


def train_perceptron(instances, lr, epochs):

    # Weight initialization
    weights = [0] * (len(instances[0])-1)

    for _ in range(epochs):
        for instance in instances:

            # Forward pass
            in_value = dot_product(weights, instance)
            output = sigmoid(in_value)

            # Error calculation
            error = instance[-1] - output

            # Weight update (gradient descent)
            for i in range(0, len(weights)):
                weights[i] += lr * error * output * (1-output) * instance[i]

    return weights

#Run

In [10]:
instances_tr = read_data("train.dat")
instances_te = read_data("test.dat")
lr = 0.005
epochs = 5
weights = train_perceptron(instances_tr, lr, epochs)
accuracy = get_accuracy(weights, instances_te)
print(f"#tr: {len(instances_tr):3}, epochs: {epochs:3}, learning rate: {lr:.3f}; "
      f"Accuracy (test, {len(instances_te)} instances): {accuracy:.1f}")

#tr: 400, epochs:   5, learning rate: 0.005; Accuracy (test, 14 instances): 71.4


#Question 1

##Ans:
We do not apply predict(weights, instance) to the training as predict predicts a binary value (0 or 1) with reference to a threshold of 0.5. The continuous probability provided by the sigmoid function (not a hard classification) is required to train the perceptron using the sigmoid activation. The rule of weight update by gradient applies to the derivative of the output of sigmoid function outputx(1 x output), and this rule will only work when the output is the real sigmoid between 0 and 1. When predict was used, then the result would be 0 or 1 and the derivative would be zero, and the weights would not update properly. Hence, it has to be trained with sigmoid(in value) rather than predict.

#Question 2

In [11]:
# Read datasets
train_data = read_data("train.dat")
test_data = read_data("test.dat")

tr_percent = [5, 10, 25, 50, 75, 100]   # percent of training data
num_epochs = [5, 10, 20, 50, 100]       # epochs
lr = [0.005, 0.01, 0.05]                # learning rate

for tr in tr_percent:

    train_size = int(len(train_data) * tr / 100)
    subset_train = train_data[:train_size]

    for learning_rate in lr:
        for epochs in num_epochs:

            weights = train_perceptron(subset_train, learning_rate, epochs)
            accuracy = get_accuracy(weights, test_data)

            print("# tr:", f"{train_size:3d}, epochs:", f"{epochs:3d}, learning rate:", learning_rate,
                  "; Accuracy (test,", len(test_data), "instances):", round(accuracy,1))

# tr:  20, epochs:   5, learning rate: 0.005 ; Accuracy (test, 14 instances): 71.4
# tr:  20, epochs:  10, learning rate: 0.005 ; Accuracy (test, 14 instances): 71.4
# tr:  20, epochs:  20, learning rate: 0.005 ; Accuracy (test, 14 instances): 71.4
# tr:  20, epochs:  50, learning rate: 0.005 ; Accuracy (test, 14 instances): 71.4
# tr:  20, epochs: 100, learning rate: 0.005 ; Accuracy (test, 14 instances): 71.4
# tr:  20, epochs:   5, learning rate: 0.01 ; Accuracy (test, 14 instances): 71.4
# tr:  20, epochs:  10, learning rate: 0.01 ; Accuracy (test, 14 instances): 71.4
# tr:  20, epochs:  20, learning rate: 0.01 ; Accuracy (test, 14 instances): 71.4
# tr:  20, epochs:  50, learning rate: 0.01 ; Accuracy (test, 14 instances): 71.4
# tr:  20, epochs: 100, learning rate: 0.01 ; Accuracy (test, 14 instances): 28.6
# tr:  20, epochs:   5, learning rate: 0.05 ; Accuracy (test, 14 instances): 71.4
# tr:  20, epochs:  10, learning rate: 0.05 ; Accuracy (test, 14 instances): 71.4
# tr:  20, 

In [ ]:
instances_tr = read_data("train.dat")
instances_te = read_data("test.dat")
tr_percent = [5, 10, 25, 50, 75, 100] # percent of the training dataset to train with
num_epochs = [5, 10, 20, 50, 100]     # number of epochs
lr_array = [0.005, 0.01, 0.05]        # learning rate

for lr in lr_array:
  for tr_size in tr_percent:
    for epochs in num_epochs:
      size =  round(len(instances_tr)*tr_size/100)
      pre_instances = instances_tr[0:size]
      weights = train_perceptron(pre_instances, lr, epochs)
      accuracy = get_accuracy(weights, instances_te)
    print(f"#tr: {len(pre_instances):0}, epochs: {epochs:3}, learning rate: {lr:.3f}; "
            f"Accuracy (test, {len(instances_te)} instances): {accuracy:.1f}")

#tr: 20, epochs: 100, learning rate: 0.005; Accuracy (test, 14 instances): 71.4
#tr: 40, epochs: 100, learning rate: 0.005; Accuracy (test, 14 instances): 71.4
#tr: 100, epochs: 100, learning rate: 0.005; Accuracy (test, 14 instances): 71.4


#Question 3

##Ans:
It is demonstrated in the experiments that the number of training data, number of epochs, and the learning rate can positively influence the performance of the perceptron model, however, not always in a strictly positive way. In most instances, a higher percentage of the training dataset would be better in terms of test accuracy, since more patterns are acquired by the model using the data. But beyond some point further improvement is minimal or even may reduce slightly as a result of noise in the new data or nonoptimal hyperparameters. This trend could be visualized with the help of the plot, in which the accuracy tends to go up with larger amounts of training data, but ultimately reaches a plateau.
##A.
No, one does not always need to train using the complete training dataset to get the maximum test accuracy. Smaller subset of clean or representative data is sometimes sufficient to give similar or even better accuracy. The inclusion of additional data can cause noise or redundant patterns that can decrease performance in a minor way.
##B.
The second run might find itself worse in its accuracy although it uses more training data due to the smaller learning rate (0.005 vs 0.050). The lower the learning rate, the slower the weight updates will be and thus the model can not find a good solution in the same amount of epochs. Thus, the hyperparameters may be stronger sources of accuracy than the mere increase in the volume of the training data.
##C.
Yes, one can attain accuracy of more than 80.0 by playing around with other hyperparameters including larger epoch numbers, varying learning rates, or varying training data proportions. These parameters can be well tuned to enable the model to arrive at a more appropriate set of weights.
##D.
There is no guarantee that this is always a good idea to train more epochs with other hyperparameters unchanged. At some stage, the model might cease to get better and it might also overfit the training data thereby lowering the performance of the test dataset. Thus, the number of epochs usually has the best generalization performance.